In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# 將專案 root 加入 python path（讓 src/ 可以 import）
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root added:", project_root)
print("✓ Imports ready")

Project root added: C:\Users\USER\Desktop\2026_japan
✓ Imports ready


In [3]:
import numpy as np
import pandas as pd

def prepare_base_df(parquet_path, remove_sentinel=True):
    df = pd.read_parquet(parquet_path).copy()
    if remove_sentinel:
        df = df[~((df["x"]==999) & (df["y"]==999))].copy()

    return df

def add_lags_and_rollings(df, seq_len, start_date="2023-01-01"):
    # weekday/is_weekend（你如果有更準的 d->date 對照，就改這裡）
    df["date"] = pd.to_datetime(start_date) + pd.to_timedelta(df["d"], unit="D")
    df["weekday"] = df["date"].dt.weekday
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)

    df = df.sort_values(["x","y","t","d"]).reset_index(drop=True)
    g = df.groupby(["x","y","t"])["count"]

    # LSTM 用：lag_1..lag_seq_len
    for k in range(1, seq_len+1):
        df[f"lag_{k}"] = g.shift(k)

    # LGBM 常用：rolling
    df["rolling_3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
    df["rolling_7"] = g.transform(lambda s: s.shift(1).rolling(7).mean())

    return df



In [4]:
import joblib
import torch
import torch.nn as nn
from src.LSTM import LSTMRegEmbed, train_lstm_embed

def load_lgbm(path):
    model = joblib.load(path)
    features = ["weekday","t","x","y","is_weekend","lag_1","lag_7","rolling_3","rolling_7"]
    return model, features

def load_lstm_embed(path):
    ckpt = torch.load(path, map_location="cpu")
    cfg = ckpt["config"]

    model = LSTMRegEmbed(
        hidden=cfg["hidden"],
        layers=cfg["layers"],
        n_weekday=cfg["n_weekday"],
        n_t=cfg["n_t"],
        n_x=cfg["n_x"],
        n_y=cfg["n_y"],
        emb_wd=cfg["emb_wd"],
        emb_t=cfg["emb_t"],
        emb_x=cfg["emb_x"],
        emb_y=cfg["emb_y"],
    )
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model, cfg

In [6]:
def make_df_pred_lgbm(df_feat, lgbm_model, lgbm_features):
    X = df_feat[lgbm_features]
    score = lgbm_model.predict(X)
    score = np.clip(score, 0, None)

    df_pred = df_feat[["d","t","x","y"]].copy()
    df_pred["score"] = score
    return df_pred

def make_df_pred_lstm_embed(df_feat, lstm_model, cfg, batch_size=4096):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    lstm_model.to(device)
    lstm_model.eval()

    seq_len = cfg["seq_len"]
    lag_cols = [f"lag_{k}" for k in range(seq_len, 0, -1)]  # 舊→新

    X_seq = df_feat[lag_cols].to_numpy(np.float32)[:, :, None]
    weekday = df_feat["weekday"].to_numpy(np.int64)
    t_id    = df_feat["t"].to_numpy(np.int64)
    x_id    = df_feat["x"].to_numpy(np.int64)
    y_id    = df_feat["y"].to_numpy(np.int64)
    is_wknd = df_feat["is_weekend"].to_numpy(np.float32)[:, None]

    preds = []
    n = len(df_feat)
    with torch.no_grad():
        for i in range(0, n, batch_size):
            xs = torch.from_numpy(X_seq[i:i+batch_size]).to(device)
            wd = torch.from_numpy(weekday[i:i+batch_size]).to(device)
            tt = torch.from_numpy(t_id[i:i+batch_size]).to(device)
            xx = torch.from_numpy(x_id[i:i+batch_size]).to(device)
            yy = torch.from_numpy(y_id[i:i+batch_size]).to(device)
            wk = torch.from_numpy(is_wknd[i:i+batch_size]).to(device)

            pred_log = lstm_model(xs, wd, tt, xx, yy, wk).cpu().numpy()  # log1p(count)
            pred = np.expm1(pred_log)                                    # 回到 count
            preds.append(pred)

    score = np.clip(np.concatenate(preds), 0, None)

    df_pred = df_feat[["d","t","x","y"]].copy()
    df_pred["score"] = score
    return df_pred

In [7]:
import numpy as np
import pandas as pd
from src.util import *
# -----------------------
# 1) Build a query set for backtesting
# -----------------------
def build_queries_from_truth(df_true: pd.DataFrame, days: list[int], K_hotspots: int = 3):
    """
    For each (d,t), pick top-K true hotspots as query centers (x0,y0).
    Returns queries: DataFrame with columns [d,t,x0,y0]
    """
    df = df_true[df_true["d"].isin(days)].copy()
    # For each (d,t), pick top-K cells by true count
    df["rank"] = df.groupby(["d","t"])["count"].rank(method="first", ascending=False)
    q = df[df["rank"] <= K_hotspots][["d","t","x","y"]].rename(columns={"x":"x0","y":"y0"})
    return q.reset_index(drop=True)

# -----------------------
# 2) Evaluate circles for each query
# -----------------------
def evaluate_circles(
    df_true: pd.DataFrame,
    df_pred: pd.DataFrame,
    queries: pd.DataFrame,
    coverages=(0.5, 0.8, 0.95),
    radius_cells: int = 8,      # neighborhood radius in GRID units (cell distance)
    cell_size_m: float | None = None,  # if you know meters per grid-cell, set here
    use_softmax: bool = False,  # if your score isn't count-like, softmax helps
    temp: float = 1.0,
):
    """
    Backtest on queries (d,t,x0,y0):
    - build candidate cells within radius_cells of (x0,y0)
    - predicted probs from df_pred
    - true probs from df_true
    - compute set-based & circle-based calibration and average radius
    """
    # Index for fast lookup
    true_map = df_true.set_index(["d","t","x","y"])["count"]
    pred_map = df_pred.set_index(["d","t","x","y"])["score"]

    rows = []
    for _, q in queries.iterrows():
        d, t, x0, y0 = int(q.d), int(q.t), int(q.x0), int(q.y0)

        # candidate grid: use cells that exist in df_pred at this (d,t), then filter by distance to center
        # (you can also precompute a full grid list if你已補0)
        pred_slice = df_pred[(df_pred["d"]==d) & (df_pred["t"]==t)][["x","y","score"]]
        if pred_slice.empty:
            continue

        cells_xy = pred_slice[["x","y"]].to_numpy()
        dist = np.hypot(cells_xy[:,0]-x0, cells_xy[:,1]-y0)
        cand_mask = dist <= radius_cells + 1e-9
        if cand_mask.sum() < 5:
            continue

        cand_xy = cells_xy[cand_mask]
        pred_scores = pred_slice["score"].to_numpy()[cand_mask]

        # true counts aligned to candidate cells
        true_counts = []
        for (x,y) in cand_xy:
            true_counts.append(true_map.get((d,t,int(x),int(y)), 0.0))
        true_counts = np.asarray(true_counts, dtype=float)

        # predicted probs
        if use_softmax:
            p_pred = softmax(pred_scores, temp=temp)
        else:
            p_pred = normalize_nonneg(pred_scores)

        # true probs
        p_true = normalize_nonneg(true_counts)

        # evaluate each alpha
        for alpha in coverages:
            idx_region = select_mass_region(cand_xy, p_pred, alpha)

            # --- set-based: true mass captured by predicted region set
            true_mass_in_set = float(p_true[idx_region].sum())

            # --- circle-based: convert region set to circle, then compute true mass in circle
            r_cells, idx_circle = circle_radius_by_mass(cand_xy, p_pred, x0, y0, alpha)
            true_mass_in_circle = float(p_true[idx_circle].sum())

            r_out = r_cells if cell_size_m is None else r_cells * cell_size_m

            rows.append({
                "d": d, "t": t, "x0": x0, "y0": y0,
                "alpha": alpha,
                "true_mass_in_set": true_mass_in_set,
                "true_mass_in_circle": true_mass_in_circle,
                "radius": r_out,
                "n_cand": int(len(cand_xy)),
                "n_set": int(len(idx_region)),
                "n_circle": int(len(idx_circle)),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out, None

    # Summary (calibration + sharpness)
    summary = out.groupby("alpha").agg(
        set_calib=("true_mass_in_set", "mean"),
        circle_calib=("true_mass_in_circle", "mean"),
        radius_mean=("radius", "mean"),
        radius_p90=("radius", lambda x: float(np.quantile(x, 0.9))),
        n=("radius", "size"),
    ).reset_index()

    # calibration error
    summary["set_calib_err"] = summary["set_calib"] - summary["alpha"]
    summary["circle_calib_err"] = summary["circle_calib"] - summary["alpha"]

    return out, summary

In [8]:
def run_calibration_for_models(
    df_true,
    df_feat,             # 含 lag/rolling/weekday/is_weekend 的特徵表
    queries,             # from build_queries_from_truth
    model_specs,         # dict: name -> callable(df_feat)->df_pred
    coverages=(0.5,0.8,0.95),
    radius_cells=8,
    use_softmax=False
):
    summaries = []
    for name, make_pred_fn in model_specs.items():
        df_pred = make_pred_fn(df_feat)

        details, summary = evaluate_circles(
            df_true=df_true,
            df_pred=df_pred,
            queries=queries,
            coverages=coverages,
            radius_cells=radius_cells,
            cell_size_m=None,
            use_softmax=use_softmax
        )

        if summary is None or summary.empty:
            continue

        summary = summary.copy()
        summary.insert(0, "model", name)
        summaries.append(summary)

    summary_all = pd.concat(summaries, ignore_index=True).sort_values(["alpha","model"])
    return summary_all


def run_one_lstm_spec(spec, df_train, df_val, df_feat_infer, df_true_eval, queries,
                     coverages=(0.5,0.8,0.95), radius_cells=8):
    # 1) 建模型
    model = LSTMRegEmbed(
        hidden=spec["hidden"],
        layers=spec["layers"],
        n_weekday=spec["n_weekday"],
        n_t=spec["n_t"],
        n_x=spec["n_x"],
        n_y=spec["n_y"],
        emb_wd=spec["emb_wd"],
        emb_t=spec["emb_t"],
        emb_x=spec["emb_x"],
        emb_y=spec["emb_y"],
    )

    # 2) 訓練（你需要一個 train_lstm_embed，和你現有 train_lstm 類似）
    model = train_lstm_embed(
        model=model,
        train_df=df_train,
        val_df=df_val,
        seq_len=spec["seq_len"],
        epochs=spec.get("epochs", 20),
        lr=spec.get("lr", 1e-3),
        batch_size=spec.get("batch_size", 2048),
        seed=spec.get("seed", 42),
    )

    # 3) 推論產 df_pred
    cfg = spec.copy()  # make_df_pred_lstm_embed 需要用 cfg["seq_len"] 等
    df_pred = make_df_pred_lstm_embed(df_feat_infer, model, cfg)

    # 4) 回測圈圈
    _, summary = evaluate_circles(
        df_true=df_true_eval,
        df_pred=df_pred,
        queries=queries,
        coverages=coverages,
        radius_cells=radius_cells,
        use_softmax=False
    )
    summary = summary.copy()
    summary.insert(0, "model", spec["name"])
    return summary

In [9]:
from src.util import densify_topk_series, split_by_day

PARQUET_PATH = "../data/processed/sapporo_density.parquet"
LGBM_PATH    = "../models/lgbm_sapporo.pkl"
LSTM_PATH    = "../models/lstm_sapporo.pkl"

# 1) 先讀資料
df_raw = prepare_base_df(PARQUET_PATH, remove_sentinel=True)
df_true = densify_topk_series(df_raw, top_k=5000)

# 2) 載 LSTM 取得 seq_len，確保特徵 lag 做到足夠長
lstm_model, cfg = load_lstm_embed(LSTM_PATH)
SEQ_LEN = cfg["seq_len"]

# 3) 做特徵（lag_1..lag_SEQ_LEN + rolling）
df_feat = add_lags_and_rollings(df_true.copy(), seq_len=SEQ_LEN, start_date="2023-01-01")

# 4) 針對「公平比較」：只用兩模型都能推論的列
need_lstm = [f"lag_{k}" for k in range(1, SEQ_LEN+1)]
need_lgbm = ["lag_1","lag_7","rolling_3","rolling_7"]
df_feat = df_feat.dropna(subset=list(set(need_lstm + need_lgbm))).copy()

train_df, val_df, test_df = split_by_day(df_feat, test_days=7, val_days=7)

# 5) 讓 df_true / queries 也對齊可推論的天（避免 query 落在太早的天）
min_day = int(df_feat["d"].min())
df_true_eval = df_true[df_true["d"] >= min_day].copy()

test_days = sorted(df_true_eval["d"].unique())[-7:]   # 你也可以改成 val/test window
queries = build_queries_from_truth(df_true_eval, test_days, K_hotspots=3)

# 6) 載 LGBM
lgbm_model, lgbm_features = load_lgbm(LGBM_PATH)

# 7) 定義「每個模型怎麼產 df_pred」
model_specs = {
    "LGBM": lambda df: make_df_pred_lgbm(df, lgbm_model, lgbm_features),
    "LSTM+Embed": lambda df: make_df_pred_lstm_embed(df, lstm_model, cfg),
}

# 8) 跑回測校準並合併 summary
summary_all = run_calibration_for_models(
    df_true=df_true_eval,
    df_feat=df_feat,
    queries=queries,
    model_specs=model_specs,
    coverages=(0.5,0.8,0.95),
    radius_cells=8
)

summary_all

,model,alpha,set_calib,circle_calib,radius_mean,radius_p90,n,set_calib_err,circle_calib_err
0,LGBM,0.50,0.514289,0.521474,2.677440,3.516897,733,0.014289,0.021474
3,LSTM+Embed,0.50,0.515527,0.517691,2.639403,3.162278,733,0.015527,0.017691
1,LGBM,0.80,0.804048,0.812150,5.105588,6.324555,733,0.004048,0.012150
4,LSTM+Embed,0.80,0.812149,0.812337,5.086907,6.324555,733,0.012149,0.012337
2,LGBM,0.95,0.954719,0.961831,6.974305,7.548640,733,0.004719,0.011831
5,LSTM+Embed,0.95,0.960071,0.961151,6.983072,7.280110,733,0.010071,0.011151


In [ ]:
base = {
    "n_weekday": 7,
    "n_t": 48,
    "n_x": int(df_feat["x"].max()) + 1,
    "n_y": int(df_feat["y"].max()) + 1,
    "epochs": 20,
    "batch_size": 2048,
    "lr": 1e-3,
    "seed": 42,
}

specs = [
    {**base, "name":"E16_H64_L1_S14", "seq_len":14, "hidden":64, "layers":1, "emb_wd":2, "emb_t":8, "emb_x":16, "emb_y":16},
    {**base, "name":"E32_H64_L1_S14", "seq_len":14, "hidden":64, "layers":1, "emb_wd":2, "emb_t":8, "emb_x":32, "emb_y":32},
    {**base, "name":"E16_H128_L1_S14","seq_len":14, "hidden":128,"layers":1, "emb_wd":2, "emb_t":8, "emb_x":16, "emb_y":16},
    {**base, "name":"E16_H64_L2_S14", "seq_len":14, "hidden":64, "layers":2, "emb_wd":2, "emb_t":8, "emb_x":16, "emb_y":16},
    {**base, "name":"E16_H64_L1_S28", "seq_len":28, "hidden":64, "layers":1, "emb_wd":2, "emb_t":8, "emb_x":16, "emb_y":16},
]

In [ ]:
summaries = []
for spec in specs:
    s = run_one_lstm_spec(spec, train_df, val_df, df_feat, df_true_eval, queries,
                          coverages=(0.5,0.8,0.95), radius_cells=8)
    summaries.append(s)

summary_lstm_tune = pd.concat(summaries, ignore_index=True)

# 只看關鍵欄位 + 排序（目標：校準接近 alpha 且 radius 小）
cols = ["model","alpha","circle_calib","circle_calib_err","radius_mean","radius_p90","n"]
summary_lstm_tune[cols].sort_values(["alpha","radius_p90","circle_calib_err"])

In [10]:
# =========================
# STEP A: 追加「分布品質」評估 (Top-k hit + NLL/Cross-Entropy)
# =========================
import numpy as np
import pandas as pd

def _normalize_nonneg_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    x = np.clip(x, 0, None)
    s = x.sum()
    return x / s if s > 0 else np.ones_like(x) / len(x)

def evaluate_rank_nll(
    df_true: pd.DataFrame,
    df_pred: pd.DataFrame,
    queries: pd.DataFrame,
    radius_cells: int = 8,
    topks=(1,5,10),
    eps: float = 1e-12,
):
    """
    在每個 query (d,t,x0,y0) 的候選窗口(<=radius_cells)內：
    - p_true: 由真實 count 正規化
    - p_pred: 由預測 score 正規化
    指標：
    - hit@k: 真實 top-1 cell 是否在預測 top-k
    - mean_rank: 真實 top-1 cell 在預測排序中的名次(1=最好)
    - nll: cross-entropy = -sum p_true * log(p_pred)
    """
    true_map = df_true.set_index(["d","t","x","y"])["count"]
    # df_pred: (d,t,x,y,score)
    pred_by_dt = {k: v for k, v in df_pred.groupby(["d","t"], sort=False)}

    rows = []
    for _, q in queries.iterrows():
        d, t, x0, y0 = int(q.d), int(q.t), int(q.x0), int(q.y0)

        sl = pred_by_dt.get((d,t), None)
        if sl is None or sl.empty:
            continue

        cells_xy = sl[["x","y"]].to_numpy()
        dist = np.hypot(cells_xy[:,0]-x0, cells_xy[:,1]-y0)
        mask = dist <= radius_cells + 1e-9
        if mask.sum() < 5:
            continue

        cand = sl.loc[mask, ["x","y","score"]]
        cand_xy = cand[["x","y"]].to_numpy()
        score = cand["score"].to_numpy(dtype=float)

        # align true counts to candidates
        true_counts = np.array([true_map.get((d,t,int(x),int(y)), 0.0) for x,y in cand_xy], dtype=float)

        p_pred = _normalize_nonneg_np(score)
        p_true = _normalize_nonneg_np(true_counts)

        # true top-1 index (within cand)
        true_top = int(np.argmax(p_true))

        # pred ranks (desc)
        order = np.argsort(-p_pred)
        # rank of true_top (1-indexed)
        rank = int(np.where(order == true_top)[0][0]) + 1

        # hit@k
        hit = {f"hit@{k}": float(true_top in order[:min(k, len(order))]) for k in topks}

        # NLL / cross-entropy
        nll = float(-(p_true * np.log(p_pred + eps)).sum())

        rows.append({
            "d": d, "t": t, "x0": x0, "y0": y0,
            "n_cand": int(len(cand_xy)),
            "nll": nll,
            "mean_rank": rank,
            **hit
        })

    detail = pd.DataFrame(rows)
    if detail.empty:
        return detail, None

    summary = detail.agg({
        "nll": "mean",
        "mean_rank": "mean",
        **{f"hit@{k}": "mean" for k in topks}
    }).to_frame().T

    # 也保留 query 數量
    summary["n_queries"] = len(detail)
    return detail, summary

def build_df_pred_cache(model_specs: dict, df_feat: pd.DataFrame):
    """避免重複推論：一次把每個模型的 df_pred 先算好。"""
    cache = {}
    for name, fn in model_specs.items():
        cache[name] = fn(df_feat)
    return cache

def run_rank_nll_for_models(
    df_true_eval: pd.DataFrame,
    df_pred_cache: dict,
    query_sets: dict,
    radius_cells: int = 8,
    topks=(1,5,10),
):
    """
    query_sets: {"hotspot": queries_df, "random": queries_df, ...}
    回傳：同一張表比較不同模型在不同 query set 下的 NLL/Top-k。
    """
    out = []
    for qname, qdf in query_sets.items():
        for mname, df_pred in df_pred_cache.items():
            _, summ = evaluate_rank_nll(
                df_true=df_true_eval,
                df_pred=df_pred,
                queries=qdf,
                radius_cells=radius_cells,
                topks=topks
            )
            if summ is None:
                continue
            row = summ.iloc[0].to_dict()
            row["model"] = mname
            row["query_set"] = qname
            out.append(row)

    return pd.DataFrame(out).sort_values(["query_set","model"]).reset_index(drop=True)

In [11]:
from IPython.display import display
# =========================
# STEP B: 生成「Random queries」並做對照 (Hotspot vs Random)
# =========================
import numpy as np
import pandas as pd

def build_random_queries_from_support(df_support: pd.DataFrame, dt_pairs: pd.DataFrame, K: int = 3, seed: int = 42):
    """
    針對指定的 (d,t) 組合，每個 (d,t) 從 df_support 裡隨機抽 K 個中心 (x0,y0)。
    這樣可保證：
    - 該 (d,t) 一定有模型可推論的格子（因為 support 來自 df_feat）
    - query 數量與 hotspot queries 可對齊
    """
    rng = np.random.default_rng(seed)
    out = []
    support_by_dt = {k: v for k, v in df_support.groupby(["d","t"], sort=False)}

    for _, r in dt_pairs.iterrows():
        d, t = int(r.d), int(r.t)
        sl = support_by_dt.get((d,t), None)
        if sl is None or sl.empty:
            continue

        xy = sl[["x","y"]].drop_duplicates().to_numpy()
        if len(xy) == 0:
            continue

        idx = rng.choice(len(xy), size=K, replace=(len(xy) < K))
        for j in idx:
            out.append({"d": d, "t": t, "x0": int(xy[j,0]), "y0": int(xy[j,1])})

    return pd.DataFrame(out)

# 1) Hotspot queries 你已經有：queries
dt_pairs = queries[["d","t"]].drop_duplicates().reset_index(drop=True)

# 2) 建 Random queries（每個 d,t 抽 K 個）
queries_random = build_random_queries_from_support(df_feat, dt_pairs, K=3, seed=42)

print("hotspot queries:", len(queries))
print("random  queries:", len(queries_random))

# 3) 先把 df_pred 都算好（避免重複推論）
df_pred_cache = build_df_pred_cache(model_specs, df_feat)

# 4) STEP A 指標：Top-k hit + NLL（對 hotspot vs random）
query_sets = {"hotspot": queries, "random": queries_random}
rank_nll_table = run_rank_nll_for_models(
    df_true_eval=df_true_eval,
    df_pred_cache=df_pred_cache,
    query_sets=query_sets,
    radius_cells=8,
    topks=(1,5,10),
)

# 5) 同時也跑 circle calibration（對 hotspot vs random）
calib_tables = []
for qname, qdf in query_sets.items():
    s = run_calibration_for_models(
        df_true=df_true_eval,
        df_feat=df_feat,
        queries=qdf,
        model_specs=model_specs,
        coverages=(0.5,0.8,0.95),
        radius_cells=8
    ).copy()
    s.insert(1, "query_set", qname)
    calib_tables.append(s)

calib_compare = pd.concat(calib_tables, ignore_index=True).sort_values(["query_set","alpha","model"])

print("=== STEP A: Distribution metrics (NLL / Top-k) ===")
display(rank_nll_table)

print("=== STEP B: Circle calibration (hotspot vs random) ===")
display(calib_compare)

hotspot queries: 889
random  queries: 945
=== STEP A: Distribution metrics (NLL / Top-k) ===


,nll,mean_rank,hit@1,hit@5,hit@10,n_queries,model,query_set
0,3.687011,1.733970,0.616644,0.976808,0.993179,733.0,LGBM,hotspot
1,3.676636,1.653479,0.694407,0.965894,0.987722,733.0,LSTM+Embed,hotspot
2,3.271641,2.470672,0.550787,0.924177,0.967096,699.0,LGBM,random
3,3.266629,2.663805,0.602289,0.892704,0.957082,699.0,LSTM+Embed,random


=== STEP B: Circle calibration (hotspot vs random) ===


,model,query_set,alpha,set_calib,circle_calib,radius_mean,radius_p90,n,set_calib_err,circle_calib_err
0,LGBM,hotspot,0.50,0.514289,0.521474,2.677440,3.516897,733,0.014289,0.021474
1,LSTM+Embed,hotspot,0.50,0.515527,0.517691,2.639403,3.162278,733,0.015527,0.017691
2,LGBM,hotspot,0.80,0.804048,0.812150,5.105588,6.324555,733,0.004048,0.012150
3,LSTM+Embed,hotspot,0.80,0.812149,0.812337,5.086907,6.324555,733,0.012149,0.012337
4,LGBM,hotspot,0.95,0.954719,0.961831,6.974305,7.548640,733,0.004719,0.011831
5,LSTM+Embed,hotspot,0.95,0.960071,0.961151,6.983072,7.280110,733,0.010071,0.011151
6,LGBM,random,0.50,0.516008,0.531394,4.505149,6.324555,699,0.016008,0.031394
7,LSTM+Embed,random,0.50,0.525003,0.534390,4.507484,6.324555,699,0.025003,0.034390
8,LGBM,random,0.80,0.807016,0.824903,6.033351,7.280110,699,0.007016,0.024903
9,LSTM+Embed,random,0.80,0.816542,0.828165,6.048238,7.280110,699,0.016542,0.028165
